In [42]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [26]:
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

model_base = pd.read_csv(PROCESSED_DATA_DIR / "rent_prediction_base.csv")
model_base.head()

,date_rent,area_code,area_name_rent,average_monthly_rent,region,date_house,area_name_house,average_house_price,area_name_key,median_annual_pay,income_confidence_pct,rent_to_income_ratio
0,2026-04-01,E06000001,Hartlepool,566,North East,2026-04-01,Hartlepool,129129,hartlepool,28830.0,6.7,0.235588
1,2026-04-01,E06000002,Middlesbrough,709,North East,2026-04-01,Middlesbrough,139005,middlesbrough,30253.0,5.5,0.281228
2,2026-04-01,E06000003,Redcar and Cleveland,644,North East,2026-04-01,Redcar and Cleveland,154422,redcar and cleveland,28630.0,6.5,0.269927
3,2026-04-01,E06000004,Stockton-on-Tees,736,North East,2026-04-01,Stockton-on-Tees,169540,stockton-on-tees,29877.0,6.9,0.295612
4,2026-04-01,E06000005,Darlington,671,North East,2026-04-01,Darlington,156880,darlington,30100.0,8.4,0.267508


In [27]:
target = "average_monthly_rent"

features = [
    "average_house_price",
    "median_annual_pay"
]


In [28]:
X = model_base[features]
y = model_base[target]

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

In [30]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [31]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_results = evaluate_model(baseline, X_test, y_test)
baseline_results

{'MAE': 360.2790178571429,
 'RMSE': np.float64(493.01255103328555),
 'R2': -0.010234330997408225}

In [32]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_results = evaluate_model(linear_model, X_test, y_test)
linear_results

{'MAE': 136.8519951600489,
 'RMSE': np.float64(171.81005228179157),
 'R2': 0.877311653929783}

In [33]:
random_forest = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

random_forest.fit(X_train, y_train)

forest_results = evaluate_model(random_forest, X_test, y_test)
forest_results

{'MAE': 161.952265625,
 'RMSE': np.float64(218.72096861015777),
 'R2': 0.8011676648977117}

In [34]:
results = pd.DataFrame([
    {"model": "Baseline mean", **baseline_results},
    {"model": "Linear regression", **linear_results},
    {"model": "Random forest", **forest_results}
])

results

,model,MAE,RMSE,R2
0,Baseline mean,360.279018,493.012551,-0.010234
1,Linear regression,136.851995,171.810052,0.877312
2,Random forest,161.952266,218.720969,0.801168


<p>A <b>simple linear regression</b> using average house price and median annual pay predicts local rent much better than guessing the average rent.</p>

In [36]:
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": random_forest.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance

,feature,importance
0,average_house_price,0.896616
1,median_annual_pay,0.103384


In [35]:
linear_predictions = linear_model.predict(X_test)

prediction_results = model_base.loc[X_test.index, [
    "area_code",
    "area_name_rent",
    "average_house_price",
    "median_annual_pay"
]].copy()
prediction_results["actual_rent"] = y_test
prediction_results["predicted_rent"] = linear_predictions
prediction_results["error"] = prediction_results["predicted_rent"] - prediction_results["actual_rent"]
prediction_results["absolute_error"] = prediction_results["error"].abs()

prediction_results.head()

,area_code,area_name_rent,average_house_price,median_annual_pay,actual_rent,predicted_rent,error,absolute_error
173,E07000175,Newark and Sherwood,236330,29806.0,789,896.404544,107.404544,107.404544
33,E06000036,Bracknell Forest,397667,36067.0,1499,1419.260873,-79.739127,79.739127
165,E07000147,North Norfolk,285303,28421.0,858,1019.482902,161.482902,161.482902
78,E07000043,North Devon,277169,28386.0,851,996.166245,145.166245,145.166245
93,E07000071,Colchester,298311,31589.0,1217,1090.521864,-126.478136,126.478136


In [37]:
prediction_results.sort_values("absolute_error", ascending=False).head(10)

,area_code,area_name_rent,average_house_price,median_annual_pay,actual_rent,predicted_rent,error,absolute_error
272,E09000012,Hackney,612501,40237.0,2615,2070.296058,-544.703942,544.703942
262,E09000002,Barking and Dagenham,360007,32911.0,1688,1278.835860,-409.164140,409.164140
280,E09000020,Kensington and Chelsea,1272760,46690.0,3597,4002.145225,405.145225,405.145225
42,E06000045,Southampton,234150,32907.0,1248,923.888222,-324.111778,324.111778
76,E07000041,Exeter,287013,29531.0,1313,1036.343139,-276.656861,276.656861
17,E06000018,Nottingham,192543,26512.0,1007,737.205356,-269.794644,269.794644
111,E07000089,Hart,473396,39341.0,1414,1668.316429,254.316429,254.316429
223,E07000243,Stevenage,319265,35747.0,1421,1194.704536,-226.295464,226.295464
82,E07000047,West Devon,307291,26556.0,837,1061.260525,224.260525,224.260525
63,E07000009,East Cambridgeshire,338848,34311.0,1018,1234.352902,216.352902,216.352902


## The most undervalued area is Hackney, where predicted rent is undervalued by £544.

In [38]:
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": linear_model.coef_
})

coefficients

,feature,coefficient
0,average_house_price,0.002820
1,median_annual_pay,0.010845


In [39]:
model_base["region"].value_counts()

region
South East                  64
East of England             45
North West                  35
East Midlands               35
London                      32
West Midlands               30
South West                  26
Wales                       22
Yorkshire and The Humber    15
North East                  12
Name: count, dtype: int64

In [40]:
numeric_features = [
    "average_house_price",
    "median_annual_pay"
]

categorical_features = [
    "region"
]

features_with_region = numeric_features + categorical_features

X_region = model_base[features_with_region]
y = model_base["average_monthly_rent"]

In [43]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features)
    ]
)